# 04 · Classification

[Index](./00_index.ipynb) · [← Previous](./03_spatial_graphs.ipynb) · [Next →](./05_gis_extraction.ipynb)

The IFC sample model `AC20-FZK-Haus.ifc` is provided as part of the KIT IFC example dataset, developed by the Institute for Automation and Applied Informatics (IAI) at Karlsruhe Institute of Technology (KIT). It is distributed for unrestricted use and commonly referenced in IFC tutorials, including IfcOpenShell documentation and sample workflows.

This notebook is standalone: it re-downloads and re-opens the model so it can run on its own in Colab, independent of the other notebooks in this portfolio.

In [1]:
!pip install -q ifcopenshell pandas requests

In [2]:
import ifcopenshell
import ifcopenshell.util.element
import pandas as pd
import requests

In [3]:
# Load the sample IFC model
import ifcopenshell
import requests

r = requests.get("https://www.ifcwiki.org/images/e/e3/AC20-FZK-Haus.ifc")
with open("model.ifc", "w") as f:
    f.write(r.text)

model = ifcopenshell.open("model.ifc")

print("Schema:", model.schema)
print("Walls:", len(model.by_type("IfcWall")))

Schema: IFC4
Walls: 13


## Classification
<a id="classification"></a>

In [4]:
# Main classes to classify
classes = [
    "IfcWall", "IfcSlab", "IfcDoor", "IfcWindow", "IfcColumn",
    "IfcBeam", "IfcStair", "IfcRamp", "IfcRoof", "IfcSpace",
]

rows = []

for cls in classes:
    for e in model.by_type(cls):
        psets = ifcopenshell.util.element.get_psets(e)

        rows.append({
            "GlobalId": e.GlobalId,
            "Class": e.is_a(),
            "Name": e.Name,
            "ObjectType": getattr(e, "ObjectType", None),
            "PredefinedType": getattr(e, "PredefinedType", None),

            "IsExternal": (
                psets.get("Pset_WallCommon", {}).get("IsExternal")
                or psets.get("Pset_DoorCommon", {}).get("IsExternal")
                or psets.get("Pset_WindowCommon", {}).get("IsExternal")
                or psets.get("Pset_SlabCommon", {}).get("IsExternal")
            ),
            "LoadBearing": (
                psets.get("Pset_WallCommon", {}).get("LoadBearing")
                or psets.get("Pset_SlabCommon", {}).get("LoadBearing")
                or psets.get("Pset_ColumnCommon", {}).get("LoadBearing")
                or psets.get("Pset_BeamCommon", {}).get("LoadBearing")
            ),
            "FireRating": (
                psets.get("Pset_WallCommon", {}).get("FireRating")
                or psets.get("Pset_DoorCommon", {}).get("FireRating")
                or psets.get("Pset_WindowCommon", {}).get("FireRating")
                or psets.get("Pset_SlabCommon", {}).get("FireRating")
            ),
        })

df_classification = pd.DataFrame(rows)
df_classification.head()

,GlobalId,Class,Name,ObjectType,PredefinedType,IsExternal,LoadBearing,FireRating
0,2XPyKWY018sA1ygZKgQPtU,IfcWallStandardCase,Wand-Int-ERDG-4,NaN,NaN,None,None,None
1,3PfS__Y_DBAfq5naM6zD2Z,IfcWallStandardCase,Wand-Int-ERDG-2,NaN,NaN,None,None,None
2,2ptk1k7qn8_Qk22vjh$0DE,IfcWallStandardCase,Wand-Int-ERDG-1,NaN,NaN,None,None,None
3,3jjW3rL656ex34Gws22EfM,IfcWallStandardCase,Wand-Int-ERDG-3,NaN,NaN,None,None,None
4,1$wmdwWPjDYuku_ghVkynE,IfcWallStandardCase,Wand-Int-ERDG-5,NaN,NaN,None,None,None


In [5]:
# Count by IFC class
df_classification.groupby("Class").size().reset_index(name="count")

,Class,count
0,IfcBeam,4
1,IfcDoor,5
2,IfcSlab,4
3,IfcSpace,7
4,IfcStair,1
5,IfcWallStandardCase,13
6,IfcWindow,11


In [6]:
# Count by class + predefined type
df_classification.groupby(
    ["Class", "PredefinedType"], dropna=False
).size().reset_index(name="count")

,Class,PredefinedType,count
0,IfcBeam,NaN,4
1,IfcDoor,NaN,5
2,IfcSlab,BASESLAB,1
3,IfcSlab,FLOOR,1
4,IfcSlab,ROOF,2
5,IfcSpace,NaN,7
6,IfcStair,NaN,1
7,IfcWallStandardCase,NaN,13
8,IfcWindow,NaN,11


In [7]:
# Load-bearing classification
df_classification.groupby(
    ["Class", "LoadBearing"], dropna=False
).size().reset_index(name="count")

,Class,LoadBearing,count
0,IfcBeam,NaN,1
1,IfcBeam,True,3
2,IfcDoor,NaN,5
3,IfcSlab,NaN,4
4,IfcSpace,NaN,7
5,IfcStair,NaN,1
6,IfcWallStandardCase,NaN,13
7,IfcWindow,NaN,11


In [8]:
# Export to CSV
df_classification.to_csv("ifc_classification.csv", index=False)